In [ ]:
def rol8(val, n):
    """Xoay trái 8-bit"""
    return ((val << (n % 8)) & 0xFF) | (val >> (8 - (n % 8)))

def ror8(val, n):
    """Xoay phải 8-bit"""
    return (val >> (n % 8)) | ((val << (8 - (n % 8))) & 0xFF)

def check_key(input_bytes):
    # offset là mảng 64 bytes từ file binary ban đầu
    
    offset = [
        0xE5, 0x5F, 0xA6, 0xFC, 0xFA, 0x04, 0xC7, 0x87, 0x6E, 0x75,
        0xB0, 0x24, 0x8F, 0x8D, 0x95, 0x13, 0xC4, 0x27, 0xF6, 0xD2,
        0x1D, 0xB6, 0xBD, 0xEE, 0x10, 0xA5, 0xFB, 0x45, 0xBB, 0x86,
        0x0C, 0xED, 0xA8, 0xA0, 0x1E, 0x4B, 0x28, 0xE3, 0xCF, 0x54,
        0x6F, 0x97, 0xF1, 0x36, 0xAF, 0xB3, 0xA3, 0x83, 0xA4, 0xF5,
        0x4A, 0x1A, 0x55, 0xA7, 0x72, 0x61, 0xB7, 0xE6, 0x29, 0x68,
        0xC2, 0x50, 0x80, 0x60
    ]

    length = len(input_bytes)
    if length == 0:
        return 1

    # Khởi tạo các biến trạng thái
    v16 = 90             # 0x5A
    v17 = ((-89) & 0xFF) # 0xA7 (unsigned 8-bit)
    v21 = 0
    v24 = (length ^ 0xC3) & 0xFF
    v22 = (v24 ^ 0x3D) & 0xFF
    
    # Làm việc trên một bản sao của mảng đầu vào (giả lập mảng bytes)
    buffer = list(input_bytes)

    # --- PASS 1: Vòng lặp từ Case 17 đến Case 35 ---
    for i in range(length):
        v18 = buffer[i]
        v19 = offset[(v16 + i) & 0x3F]
        v20 = offset[(5 * i + 7) & 0x3F]

        # Biến đổi v18 dựa trên điều kiện
        if ((i ^ v24) & 1) != 0:
            v18 = ((v18 + v19) ^ v17) & 0xFF
        else:
            v18 = ((v20 ^ v18) + (v16 & 0xF)) & 0xFF

        # Xoay bit (Rotation)
        if (v18 & 1) != 0:
            v18 = ror8(v18, 2)
        else:
            v18 = rol8(v18, 3)

        v18 = (v18 ^ (13 * i)) & 0xFF
        v18 = (v18 + v22) & 0xFF
        buffer[i] = v18

        # Cập nhật các biến điều khiển luồng
        v21 = (v21 ^ (buffer[i] + i + (v24 & 3))) & 0xFF
        v16 = (v17 ^ (v16 + v21 + 17)) & 0xFF
        v17 = ((v17 ^ offset[(i + 11) & 0x3F]) + 3) & 0xFF

        if ((v21 + i) & 3) == 0:
            v16 = (v16 ^ 0x5C) & 0xFF
        else:
            v16 = (v16 ^ 0xA1) & 0xFF

    # --- PASS 2: Vòng lặp từ Case 65 đến Case 79 ---
    for i in range(length):
        v19 = buffer[i]

        # Xoay bit
        if (i % 3) != 0:
            v19 = rol8(v19, 1)
        else:
            v19 = ror8(v19, 1)

        v19 = (v19 ^ (i + v16)) & 0xFF

        # Biến đổi v19 theo offset
        cond_val = (i + v24 + v21) & 1
        off_idx = (9 * i + 1) & 0x3F
        if cond_val != 0:
            v19 = (v19 + (v22 ^ offset[off_idx])) & 0xFF
        else:
            v19 = (v19 ^ (offset[off_idx] + v22)) & 0xFF

        buffer[i] = v19
        v21 = ((v16 + v21 + buffer[i]) ^ i) & 0xFF

        if ((i ^ v21) & 3) == 2:
            v22 = (v22 ^ (buffer[i] + 51)) & 0xFF
        else:
            v22 = (v22 ^ (buffer[i] ^ 0xC7)) & 0xFF

    # --- PASS 3: Vòng lặp từ Case 113 đến Case 122 ---
    # Vòng lặp này xử lý cặp ký tự từ hai đầu (nhưng nhảy bước 2)
    for i in range(0, length - 1, 2):
        v26 = length - i - 1
        v18 = buffer[i]
        v19 = buffer[v26]

        if ((i + v24 + v22) & 1) != 0:
            v18 = (v18 ^ v16) & 0xFF
        else:
            v19 = (v19 ^ v21) & 0xFF

        buffer[i] = (v19 ^ offset[(v17 + i) & 0x3F]) & 0xFF
        buffer[v26] = (v18 ^ offset[(v16 + v26) & 0x3F]) & 0xFF

    # Final logic (Case 144)
    # Dù kết quả cuối cùng của hàm gốc là return 1, các biến này vẫn được cập nhật
    if (((v22 ^ v16 ^ v21) ^ (v24 + length)) & 1) != 0:
        v21 ^= 0xAA
        v16 ^= 0x13
        v22 ^= 0x5B

    return 1, buffer

# Ví dụ sử dụng:
input_data = b"your_flag_here"
result_code, transformed_buffer = check_key(input_data)
print(result_code,f"Transformed: {transformed_buffer}")

1 Transformed: [133, 252, 235, 95, 42, 133, 119, 209, 124, 191, 137, 170, 19, 105]
